In [1]:
import torch
import torch.nn as nn
import math

# -----------------------------
# Positional Encoding
# -----------------------------
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)

        div_term = torch.exp(torch.arange(0, d_model, 2).float() *
                             (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)   # shape: (1, max_len, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return x


# -----------------------------
# Transformer Model
# -----------------------------
class SimpleTransformer(nn.Module):
    def __init__(self, input_dim, d_model, nhead, num_layers, output_dim, max_len=100):
        super(SimpleTransformer, self).__init__()

        self.embedding = nn.Embedding(input_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model, max_len)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=128,
            dropout=0.1,
            batch_first=True
        )

        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.fc_out = nn.Linear(d_model, output_dim)

    def forward(self, src):
        # src shape: (batch_size, seq_len)
        x = self.embedding(src) * math.sqrt(self.embedding.embedding_dim)
        x = self.pos_encoder(x)
        x = self.transformer_encoder(x)
        out = self.fc_out(x)
        return out


# -----------------------------
# Example Usage
# -----------------------------
if __name__ == "__main__":
    input_dim = 20      # vocabulary size
    d_model = 32        # embedding dimension
    nhead = 4           # number of attention heads
    num_layers = 2      # number of encoder layers
    output_dim = 20     # output vocabulary size
    seq_len = 5
    batch_size = 2

    model = SimpleTransformer(input_dim, d_model, nhead, num_layers, output_dim)

    # Example input: batch of 2 sequences, each of length 5
    src = torch.tensor([
        [1, 5, 7, 3, 2],
        [4, 6, 8, 9, 1]
    ])

    output = model(src)

    print("Input shape:", src.shape)
    print("Output shape:", output.shape)
    print("Output tensor:\n", output)

Input shape: torch.Size([2, 5])
Output shape: torch.Size([2, 5, 20])
Output tensor:
 tensor([[[-0.3968,  0.8939, -1.3547,  0.5681, -1.4595,  0.5150,  0.0871,
          -1.0908,  0.4760,  0.6782, -0.2867, -1.1502,  0.1446, -0.4549,
           0.1285,  0.6699, -0.5241,  0.5352, -0.0630, -0.7121],
         [-0.1649,  1.0212, -0.1142,  0.1144,  0.1620, -0.3454, -0.0984,
           0.8748, -0.3540, -0.5781, -0.7457, -0.4008,  0.8917, -0.0547,
           0.6804,  0.5384,  0.1025, -0.4485,  0.1016, -0.5109],
         [ 0.1705, -0.1004,  0.0286, -0.3712,  0.8649, -0.5928, -0.2817,
           0.0408,  0.1364,  0.3858,  0.1028, -0.1375, -0.6000,  1.0193,
           0.8686, -0.0484,  0.6127, -0.8442,  0.4036, -0.2005],
         [-0.4242, -0.2224, -0.7009, -0.3589, -1.0138, -0.0562, -0.8440,
          -0.3334, -1.1149,  0.2287, -0.0194,  0.4378, -0.4730,  0.5109,
          -0.0350, -0.3606,  0.5203, -0.4837,  0.8161, -0.0852],
         [-0.4818,  0.5508, -1.0572,  0.1642, -0.4956, -0.2041,  0.4786